In [1]:
import os
from nwtrace import NWTrace
import pandas as pd
import geopandas as gpd
from nwtrace.utils import verify_network_geometry

network_path = "data/more/full_sewers.geojson"

multiple = True
upstream_only = True
downstream_only = False
verbose = True

sewer_id_field = 'FACILITYID'
upstream_field = 'FROMMH'
downstream_field = 'TOMH'

outfall_file = 'data/more/BC_outfalls.csv'
id_field = 'Asset Identification'

In [2]:

outfalls = pd.read_csv(outfall_file)[id_field].tolist()

target_endpoints = outfalls
# target_endpoints = ["OF3729806115"]

outputname_extra = "BC_inlets"
output_dir = f"./out"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

result = []

In [3]:

sewershed = NWTrace(
    network=network_path,
    id_field=sewer_id_field,
    upstream_field = upstream_field,
    downstream_field = downstream_field,
    verbose=verbose,
    output_dir=output_dir,
)


Loading network file: `data\more\full_sewers.geojson`...


In [4]:

# # additional connections
fittings = gpd.read_file("data/more/fitting_connections.geojson")
nodes_up = (fittings[["FACILITYID", "TO_FIXED"]]
            .dropna(subset=["FACILITYID", "TO_FIXED"]) # remove rows with None values
            .rename(columns={"FACILITYID": 'node_id', "TO_FIXED": 'segment_id'})
            .to_dict(orient="records"))

sewershed.add_upstream_nodes(nodes_up)

catchbasin_leads = gpd.read_file("data/more/catchbasin_leads.gpkg")

new_segs = (catchbasin_leads[["FACILITYID", "UP_ASSET_ID", "DN_ASSET_ID"]]
            .rename(columns={"FACILITYID": 'segment_id', "UP_ASSET_ID": 'from', "DN_ASSET_ID": 'to'})
            .to_dict(orient="records"))

sewershed.add_segments(new_segs)

Added 5245 node-segment connection(s)
Created 627 new node(s)
Created 374 new segment(s).

Added 119614 node-segment connection(s)
Created 121055 new node(s)
Created 120788 new segment(s).



In [5]:
if multiple == False:
    result = sewershed.trace_sewershed(
        target_endpoints[0], 
        upstream_only=upstream_only, 
        downstream_only=downstream_only
    )
else:
    result = sewershed.trace_sewersheds(
        target_endpoints, 
        upstream_only=upstream_only, 
        downstream_only=downstream_only, 
    )

Tracing Sewer Network from endpoint(s) [OF3783406347(...)]
	Direction(s): upstream
Preparing directional node connection tree...
Searching Network:


100%|██████████| 190/190 [00:00<00:00, 5487.36it/s]


Found 14 connections overall to all 190 endpoints
Finished!


In [6]:
d_node, d_seg = sewershed.get_directional_lookup_tables()

In [11]:
d_seg['CL9478']

{'to': ['MH3836707104'], 'from': ['CB3838007097']}

In [9]:
d_node['MH3873708965']

{'in': ['SL51551', 'SL51542', 'SL53208'],
 'out': ['SL53230',
  'CL51548',
  'CL51549',
  'CL51553',
  'CL51554',
  'CL51556',
  'CL51557',
  'CL51399',
  'CL51547',
  'CL491']}